In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
base = '/content/drive/MyDrive/crowd-risk-lstm'
os.makedirs(f'{base}/data/raw/crowdhuman', exist_ok=True)
print("Drive mounted")

Mounted at /content/drive
Drive mounted


In [ ]:
import os
os.makedirs('/root/.kaggle', exist_ok=True)

# Paste your username and key here
kaggle_username = "jeevithasrinivas"  # ← your Kaggle username
kaggle_key = "KGAT_095edc1b2a04bbb690a9c1ec18eae7ed"         # ← your API token/key

with open('/root/.kaggle/kaggle.json', 'w') as f:
    f.write(f'{{"username":"{kaggle_username}","key":"{kaggle_key}"}}')

os.system('chmod 600 /root/.kaggle/kaggle.json')
print("Done")

Done


In [ ]:
# Search available crowd datasets
!kaggle datasets list -s "crowdhuman"


ref                                     title                                         size  lastUpdated                 downloadCount  voteCount  usabilityRating  
--------------------------------------  -------------------------------------  -----------  --------------------------  -------------  ---------  ---------------  
permanalwep/crowdhuman-crowd-detection  CrowdHuman Crowd Detection               870738309  2023-08-10 13:36:12.443000           1446          8  0.5625           
leducnhuan/crowdhuman                   CrowdHuman                             10891866902  2024-01-10 10:40:06.387000            689          5  0.3125           
ishantchintapatla/corevqa               COREVQA                                 6073786758  2025-07-23 00:21:11.603000           1390         23  0.9411765        
loctran0941/crowdhuman                  CrowdHuman                             10892046950  2025-12-19 15:02:08.767000            196          1  0.4375           
permanalwep/crow

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
base = '/content/drive/MyDrive/crowd-risk-lstm'
os.makedirs(f'{base}/data/raw', exist_ok=True)
os.makedirs(f'{base}/models', exist_ok=True)
os.makedirs(f'{base}/outputs/figures', exist_ok=True)
os.makedirs(f'{base}/outputs/reports', exist_ok=True)

%cd {base}/data/raw
print("Ready")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/crowd-risk-lstm/data/raw
Ready


In [ ]:
!pip install roboflow -q

from roboflow import Roboflow
rf = Roboflow(api_key="k8Zy8xAJBZa9efvsXFa3")
project = rf.workspace("mot20").project("crowdhuman-44xdq")
version = project.version(1)
dataset = version.download("yolov8")

print("Downloaded to:", dataset.location)

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to crowdhuman-1 in yolov8::  66%|██████▌   | 25389/38750 [06:29<02:57, 75.33it/s]

In [ ]:
import os

loc = "/content/drive/MyDrive/crowd-risk-lstm/data/raw/crowdhuman-1"

# Show data.yaml
with open(f'{loc}/data.yaml') as f:
    print("=== data.yaml ===")
    print(f.read())

# Count files
for split in ['train', 'valid', 'test']:
    img_dir = f'{loc}/{split}/images'
    lbl_dir = f'{loc}/{split}/labels'
    if os.path.exists(img_dir):
        imgs = len(os.listdir(img_dir))
        lbls = len(os.listdir(lbl_dir))
        print(f"{split}: {imgs} images, {lbls} labels")

# Peek at one label
sample = os.listdir(f'{loc}/train/labels')[0]
print(f"\nSample label ({sample}):")
with open(f'{loc}/train/labels/{sample}') as f:
    print(f.read()[:200])

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/crowd-risk-lstm/data/raw/crowdhuman-1/data.yaml'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

base = '/content/drive/MyDrive/crowd-risk-lstm/data/raw'
print("Contents of data/raw:")
for item in os.listdir(base):
    print(" ", item)

Contents of data/raw:
  sample.mp4
  crowdhuman-1


In [ ]:
import os

loc = "/content/drive/MyDrive/crowd-risk-lstm/data/raw/crowdhuman-1"

print("Top-level contents:")
for item in os.listdir(loc):
    print(" ", item)

print("\n=== data.yaml ===")
with open(f'{loc}/data.yaml') as f:
    print(f.read())

print("=== File counts ===")
for split in ['train', 'valid', 'test']:
    img_dir = f'{loc}/{split}/images'
    lbl_dir = f'{loc}/{split}/labels'
    if os.path.exists(img_dir):
        print(f"{split}: {len(os.listdir(img_dir))} images, {len(os.listdir(lbl_dir))} labels")

sample = os.listdir(f'{loc}/train/labels')[0]
print(f"\n=== Sample label: {sample} ===")
with open(f'{loc}/train/labels/{sample}') as f:
    print(f.read()[:200])

Top-level contents:
  roboflow.zip
  README.dataset.txt
  README.roboflow.txt
  data.yaml
  test
  train

=== data.yaml ===
train: ../train/images
val: ../valid/images
test: ../test/images

nc: 1
names: ['person']

roboflow:
  workspace: mot20
  project: crowdhuman-44xdq
  version: 1
  license: CC BY 4.0
  url: https://universe.roboflow.com/mot20/crowdhuman-44xdq/dataset/1
=== File counts ===


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/crowd-risk-lstm/data/raw/crowdhuman-1/train/labels'

In [ ]:
import os

loc = "/content/drive/MyDrive/crowd-risk-lstm/data/raw/crowdhuman-1"

# See what partially extracted
for folder in ['train', 'test']:
    p = f'{loc}/{folder}'
    if os.path.exists(p):
        print(f"{folder}/: {os.listdir(p)}")

# Check zip size
print(f"\nroboflow.zip size: {os.path.getsize(f'{loc}/roboflow.zip') / 1e9:.2f} GB")


train/: ['images']
test/: ['images', 'labels']

roboflow.zip size: 5.01 GB


In [ ]:
import zipfile, os, time

loc = "/content/drive/MyDrive/crowd-risk-lstm/data/raw/crowdhuman-1"

start = time.time()
with zipfile.ZipFile(f'{loc}/roboflow.zip', 'r') as z:
    total = len(z.namelist())
    print(f"Extracting {total} files...")
    z.extractall(loc)

print(f"Done in {(time.time()-start)/60:.1f} minutes")

Extracting 38750 files...


KeyboardInterrupt: 

In [ ]:
import zipfile, time, shutil, os

src_zip = "/content/drive/MyDrive/crowd-risk-lstm/data/raw/crowdhuman-1/roboflow.zip"
local = "/content/crowdhuman"
dest = "/content/drive/MyDrive/crowd-risk-lstm/data/raw/crowdhuman-1"

# Extract to fast local disk
start = time.time()
os.makedirs(local, exist_ok=True)
with zipfile.ZipFile(src_zip) as z:
    z.extractall(local)
print(f"Local extract: {(time.time()-start)/60:.1f} min")

# Verify locally first
for split in ['train', 'valid', 'test']:
    ip, lp = f'{local}/{split}/images', f'{local}/{split}/labels'
    if os.path.exists(ip):
        print(f"{split}: {len(os.listdir(ip))} imgs, {len(os.listdir(lp))} lbls")

Local extract: 2.1 min
train: 13558 imgs, 13558 lbls
valid: 3874 imgs, 3874 lbls
test: 1937 imgs, 1937 lbls


In [ ]:
import yaml, os

local = "/content/crowdhuman"

cfg = {
    'train': f'{local}/train/images',
    'val':   f'{local}/valid/images',
    'test':  f'{local}/test/images',
    'nc': 1,
    'names': ['person']
}

with open(f'{local}/data.yaml', 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

print(open(f'{local}/data.yaml').read())

names:
- person
nc: 1
test: /content/crowdhuman/test/images
train: /content/crowdhuman/train/images
val: /content/crowdhuman/valid/images



In [ ]:
import os

sample = os.listdir(f'{local}/train/labels')[0]
print(f"Sample: {sample}\n")
with open(f'{local}/train/labels/{sample}') as f:
    lines = f.read().strip().split('\n')
print(f"{len(lines)} persons in this image")
print("First 5 boxes (class x_center y_center width height):")
for l in lines[:5]:
    print(" ", l)

Sample: 273275-ed6e7000c701f067_jpg.rf.f9006bb7a59a34cec024b52980641ea2.txt

21 persons in this image
First 5 boxes (class x_center y_center width height):
  0 0.03916666666666667 0.6355555555555555 0.07833333333333334 0.7288888888888889
  0 0.07583333333333334 0.6866666666666666 0.15166666666666667 0.6266666666666667
  0 0.13333333333333333 0.6833333333333333 0.13 0.6111111111111112
  0 0.2125 0.6977777777777778 0.16166666666666665 0.6044444444444445
  0 0.25583333333333336 0.6955555555555556 0.155 0.5688888888888889


In [ ]:
!pip install ultralytics -q

from ultralytics import YOLO
import torch, os, shutil

print("GPU:", torch.cuda.get_device_name(0))

local = "/content/crowdhuman"
drive_base = "/content/drive/MyDrive/crowd-risk-lstm"

model = YOLO("yolov8s.pt")

results = model.train(
    data=f'{local}/data.yaml',
    epochs=60,
    imgsz=640,
    batch=16,
    device=0,
    freeze=10,
    optimizer='AdamW',
    lr0=0.001,
    patience=12,
    save_period=5,
    project=f'{drive_base}/outputs/yolo_training',
    name='crowdhuman_yolov8s',
    exist_ok=True,
    plots=True,
    workers=2
)

best = f'{drive_base}/outputs/yolo_training/crowdhuman_yolov8s/weights/best.pt'
shutil.copy(best, f'{drive_base}/models/finetuned_yolov8s_cctv.pt')
print("\nSaved to models/finetuned_yolov8s_cctv.pt")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 69.9 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
GPU: Tesla T4
Ultralytics 8.4.115 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/crowdhuman/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import zipfile, yaml, time
start = time.time()

with zipfile.ZipFile('/content/drive/MyDrive/crowd-risk-lstm/data/raw/crowdhuman-1/roboflow.zip') as z:
    z.extractall('/content/crowdhuman')

cfg = {'train':'/content/crowdhuman/train/images',
       'val':'/content/crowdhuman/valid/images',
       'test':'/content/crowdhuman/test/images',
       'nc':1, 'names':['person']}
with open('/content/crowdhuman/data.yaml','w') as f:
    yaml.dump(cfg, f)

print(f"Ready in {(time.time()-start)/60:.1f} min")

Mounted at /content/drive
Ready in 1.3 min


In [ ]:
!pip install ultralytics -q
from ultralytics import YOLO

ckpt = '/content/drive/MyDrive/crowd-risk-lstm/outputs/yolo_training/crowdhuman_yolov8s/weights/last.pt'
model = YOLO(ckpt)
model.train(resume=True)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 22.9 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.115 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/crowdhuman/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_mode

In [ ]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

False no GPU


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
base = '/content/drive/MyDrive/crowd-risk-lstm'
bak = '/content/drive/MyDrive/crowd-risk-lstm-BACKUP'
os.makedirs(bak, exist_ok=True)

# the irreplaceable things
for f in ['models/finetuned_yolov8s_cctv.pt',
          'outputs/yolo_training/crowdhuman_yolov8s/weights/best.pt',
          'outputs/yolo_training/crowdhuman_yolov8s/weights/last.pt',
          'data/umn_groundtruth.csv',
          'data/features/umn_features.npy',
          'data/forecast_dataset.npz']:
    src = f'{base}/{f}'
    if os.path.exists(src):
        dst = f'{bak}/{os.path.basename(f)}'
        shutil.copy(src, dst)
        print("backed up:", f)

Mounted at /content/drive
backed up: models/finetuned_yolov8s_cctv.pt
backed up: outputs/yolo_training/crowdhuman_yolov8s/weights/best.pt
backed up: outputs/yolo_training/crowdhuman_yolov8s/weights/last.pt
backed up: data/umn_groundtruth.csv
backed up: data/features/umn_features.npy
backed up: data/forecast_dataset.npz


In [ ]:
work = '/content/drive/MyDrive/for-friend'
os.makedirs(f'{work}/weights', exist_ok=True)
shutil.copy(f'{base}/outputs/yolo_training/crowdhuman_yolov8s/weights/last.pt', f'{work}/weights/last.pt')
shutil.copy(f'{base}/data/raw/crowdhuman-1/roboflow.zip', f'{work}/roboflow.zip')
print("ready to share:", work)

ready to share: /content/drive/MyDrive/for-friend


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
base = '/content/drive/MyDrive/crowd-risk-lstm/data/raw/shanghaitech'
os.makedirs(base, exist_ok=True)
%cd {base}

# Test part first — this is the one with anomalies and frame-level labels
!kaggle datasets download -d nikanvasei/shanghaitech-campus-dataset-test --unzip

# Inspect what arrived before pulling the much larger train part
import os
for root, dirs, files in os.walk('.'):
    lvl = root.count(os.sep)
    if lvl > 3: continue
    print('  '*lvl + os.path.basename(root) + '/')
    for f in files[:4]:
        print('  '*(lvl+1) + f)
    if len(files) > 4:
        print('  '*(lvl+1) + f'... +{len(files)-4} more')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/crowd-risk-lstm/data/raw/shanghaitech
Dataset URL: https://www.kaggle.com/datasets/nikanvasei/shanghaitech-campus-dataset-test
License(s): MIT
100% 8.72G/8.72G [01:30<00:00, 104MB/s]

./
  SHANGHAI/
    SHANGHAI_Test/
      SHANGHAI_test.txt
      frames/
      label/
        01_001.npy
        01_0015.npy
        01_0025.npy
        01_0027.npy
        ... +195 more


In [ ]:
import os

base = '/content/drive/MyDrive/crowd-risk-lstm/data/raw/shanghaitech'

for root, dirs, files in os.walk(base):
    depth = root.replace(base, '').count(os.sep)
    if depth > 3:
        dirs[:] = []          # stop descending
        continue
    print(f"{'  '*depth}{os.path.basename(root) or 'shanghaitech'}/   [{len(dirs)} dirs, {len(files)} files]")
    for f in files[:5]:
        print(f"{'  '*(depth+1)}{f}")
    if len(files) > 5:
        print(f"{'  '*(depth+1)}... +{len(files)-5} more")

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
print(os.listdir('/content/drive/MyDrive/crowd-risk-lstm/data/raw'))

Mounted at /content/drive
['sample.mp4', 'crowdhuman-1', 'umn', 'shanghaitech']


In [ ]:
import os

base = '/content/drive/MyDrive/crowd-risk-lstm/data/raw/shanghaitech'

for root, dirs, files in os.walk(base):
    depth = root.replace(base, '').count(os.sep)
    if depth > 3:
        dirs[:] = []
        continue
    print(f"{'  '*depth}{os.path.basename(root) or 'shanghaitech'}/   [{len(dirs)} dirs, {len(files)} files]")
    for f in files[:5]:
        print(f"{'  '*(depth+1)}{f}")
    if len(files) > 5:
        print(f"{'  '*(depth+1)}... +{len(files)-5} more")

shanghaitech/   [1 dirs, 1 files]
  shanghaitech-campus-dataset-test.zip
  SHANGHAI/   [1 dirs, 0 files]
    SHANGHAI_Test/   [1 dirs, 1 files]
      SHANGHAI_test.txt
      frames/   [119 dirs, 0 files]


In [ ]:
import os

base = '/content/drive/MyDrive/crowd-risk-lstm/data/raw/shanghaitech'

for root, dirs, files in os.walk(base):
    depth = root.replace(base, '').count(os.sep)
    if depth > 3:
        dirs[:] = []
        continue
    print(f"{'  '*depth}{os.path.basename(root) or 'shanghaitech'}/   [{len(dirs)} dirs, {len(files)} files]")
    for f in files[:5]:
        print(f"{'  '*(depth+1)}{f}")
    if len(files) > 5:
        print(f"{'  '*(depth+1)}... +{len(files)-5} more")

shanghaitech/   [1 dirs, 1 files]
  shanghaitech-campus-dataset-test.zip
  SHANGHAI/   [1 dirs, 0 files]
    SHANGHAI_Test/   [1 dirs, 1 files]
      SHANGHAI_test.txt
      frames/   [119 dirs, 0 files]


In [ ]:
import zipfile, os, time

src = '/content/drive/MyDrive/crowd-risk-lstm/data/raw/shanghaitech/shanghaitech-campus-dataset-test.zip'
local = '/content/shanghaitech'

print(f"zip size: {os.path.getsize(src)/1e9:.2f} GB")

start = time.time()
os.makedirs(local, exist_ok=True)
with zipfile.ZipFile(src) as z:
    print(f"{len(z.namelist())} entries")
    z.extractall(local)
print(f"Extracted in {(time.time()-start)/60:.1f} min")

# find where things landed
for root, dirs, files in os.walk(local):
    d = root.replace(local,'').count(os.sep)
    if d > 3:
        dirs[:] = []
        continue
    print(f"{'  '*d}{os.path.basename(root) or 'root'}/  [{len(dirs)}d {len(files)}f]")
    for f in files[:3]:
        print(f"{'  '*(d+1)}{f}")

zip size: 9.36 GB
142260 entries
Extracted in 4.1 min
shanghaitech/  [1d 0f]
  SHANGHAI/  [1d 0f]
    SHANGHAI_Test/  [2d 1f]
      SHANGHAI_test.txt
      frames/  [199d 0f]
      label/  [0d 199f]
        09_002.npy
        06_028.npy
        03_005.npy


In [ ]:
import numpy as np, os, glob
from collections import Counter

root = '/content/shanghaitech/SHANGHAI/SHANGHAI_Test'

print("=== SHANGHAI_test.txt ===")
with open(f'{root}/SHANGHAI_test.txt') as f:
    lines = f.read().strip().split('\n')
print(f"{len(lines)} lines")
for l in lines[:8]: print(" ", l)

labs = sorted(glob.glob(f'{root}/label/*.npy'))
print(f"\n=== labels ({len(labs)}) ===")
rates, lens = [], []
for p in labs:
    a = np.load(p)
    rates.append(float(a.mean())); lens.append(len(a))
for p in labs[:3]:
    a = np.load(p)
    print(f"{os.path.basename(p)}: shape {a.shape}, dtype {a.dtype}, uniq {np.unique(a)}, pos {a.mean():.3f}")

rates, lens = np.array(rates), np.array(lens)
print(f"\nframes/video: min {lens.min()} median {int(np.median(lens))} max {lens.max()} total {lens.sum()}")
print(f"positive rate: min {rates.min():.3f} median {np.median(rates):.3f} max {rates.max():.3f}")
print(f"videos with zero positives: {(rates==0).sum()}")
print(f"overall positive frames: {int((rates*lens).sum())} / {lens.sum()} ({100*(rates*lens).sum()/lens.sum():.1f}%)")

fdirs = sorted(os.listdir(f'{root}/frames'))
sample = sorted(os.listdir(f'{root}/frames/{fdirs[0]}'))
print(f"\n=== frames ===")
print(f"{fdirs[0]}: {len(sample)} files, e.g. {sample[:3]}")
print("label/frame count match:", len(sample) == len(np.load(f'{root}/label/{fdirs[0]}.npy')))

sc = Counter(d.split('_')[0] for d in fdirs)
print("\nvideos per scene:", dict(sorted(sc.items())))

=== SHANGHAI_test.txt ===
199 lines
  SHANGHAI_TEST/frames/01_001 765 0 0 765
  SHANGHAI_TEST/frames/01_0015 433 1 86 373
  SHANGHAI_TEST/frames/01_0025 601 1 148 202
  SHANGHAI_TEST/frames/01_0027 409 1 77 151
  SHANGHAI_TEST/frames/01_0028 457 1 266 330
  SHANGHAI_TEST/frames/01_004 770 0 0 770
  SHANGHAI_TEST/frames/01_0051 337 1 98 325
  SHANGHAI_TEST/frames/01_0052 337 1 119 231

=== labels (199) ===
01_001.npy: shape (765,), dtype float64, uniq [0.], pos 0.000
01_0015.npy: shape (433,), dtype uint8, uniq [0 1], pos 0.665
01_0025.npy: shape (601,), dtype uint8, uniq [0 1], pos 0.092

frames/video: min 193 median 734 max 1852 total 142060
positive rate: min 0.000 median 0.000 max 1.000
videos with zero positives: 155
overall positive frames: 7962 / 142060 (5.6%)

=== frames ===
01_001: 765 files, e.g. ['0001.jpg', '0002.jpg', '0003.jpg']
label/frame count match: True

videos per scene: {'01': 57, '02': 8, '03': 6, '04': 13, '05': 35, '06': 16, '07': 6, '08': 29, '09': 4, '10': 7, '

In [ ]:
import numpy as np, glob, os
from collections import Counter, defaultdict

root = '/content/shanghaitech/SHANGHAI/SHANGHAI_Test'

rows = []
with open(f'{root}/SHANGHAI_test.txt') as f:
    for l in f.read().strip().split('\n'):
        p = l.split()
        rows.append({'vid': os.path.basename(p[0]), 'n': int(p[1]),
                     'anom': int(p[2]), 'start': int(p[3]), 'end': int(p[4])})

anom = [r for r in rows if r['anom'] == 1]
print(f"anomalous videos: {len(anom)} / {len(rows)}")

per_scene = defaultdict(lambda: [0,0])
for r in rows:
    s = r['vid'].split('_')[0]
    per_scene[s][0] += 1
    per_scene[s][1] += r['anom']

print(f"\n{'scene':>6} {'videos':>7} {'anomalous':>10}")
for s in sorted(per_scene):
    t, a = per_scene[s]
    print(f"{s:>6} {t:>7} {a:>10}")

# event durations from start/end
durs = [r['end']-r['start'] for r in anom]
print(f"\nevent duration frames: min {min(durs)} median {int(np.median(durs))} max {max(durs)}")
print(f"at 24fps: median {np.median(durs)/24:.1f}s")

# how much normal lead-in before each event
leads = [r['start'] for r in anom]
print(f"lead-in before event: min {min(leads)} median {int(np.median(leads))} max {max(leads)}")
print(f"videos with <150 frames lead-in: {sum(1 for l in leads if l < 150)}")

anomalous videos: 44 / 199

 scene  videos  anomalous
    01      57         16
    02       8          1
    03       6          4
    04      13          4
    05      35          3
    06      16          2
    07       6          3
    08      29          5
    09       4          0
    10       7          2
    11       4          0
    12      11          4
    13       3          0

event duration frames: min 51 median 161 max 526
at 24fps: median 6.7s
lead-in before event: min 0 median 92 max 426
videos with <150 frames lead-in: 32


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import zipfile, time
src = '/content/drive/MyDrive/crowd-risk-lstm/data/raw/shanghaitech/shanghaitech-campus-dataset-test.zip'
start = time.time()
with zipfile.ZipFile(src) as z:
    z.extractall('/content/shanghaitech')
print(f"done in {(time.time()-start)/60:.1f} min")

Mounted at /content/drive
done in 3.0 min


In [ ]:
import os, json
root = '/content/shanghaitech/SHANGHAI/SHANGHAI_Test'
rows = []
with open(f'{root}/SHANGHAI_test.txt') as f:
    for l in f.read().strip().split('\n'):
        p = l.split()
        rows.append({'vid': os.path.basename(p[0]), 'n': int(p[1]),
                     'anom': int(p[2]), 'start': int(p[3]), 'end': int(p[4])})

out = '/content/drive/MyDrive/crowd-risk-lstm/data/shanghaitech_meta.json'
with open(out, 'w') as f:
    json.dump(rows, f)
print("saved", out)

saved /content/drive/MyDrive/crowd-risk-lstm/data/shanghaitech_meta.json


In [ ]:
import json

with open('/content/drive/MyDrive/crowd-risk-lstm/data/shanghaitech_meta.json') as f:
    rows = json.load(f)
anom = [r for r in rows if r['anom'] == 1]

SEQ = 30
for stride in [5, 4, 3, 2, 1]:
    span = (SEQ - 1) * stride
    usable = [r for r in anom if r['start'] >= span]
    n_samples = sum(r['start'] - span + 1 for r in usable)
    print(f"stride {stride}: span {span:3d}f ({span/24:.1f}s) | "
          f"{len(usable):2d}/44 videos | {n_samples:5d} pre-onset samples")

stride 5: span 145f (6.0s) | 13/44 videos |  1109 pre-onset samples
stride 4: span 116f (4.8s) | 18/44 videos |  1576 pre-onset samples
stride 3: span  87f (3.6s) | 23/44 videos |  2169 pre-onset samples
stride 2: span  58f (2.4s) | 33/44 videos |  3017 pre-onset samples
stride 1: span  29f (1.2s) | 37/44 videos |  4055 pre-onset samples


In [ ]:
import json
from collections import Counter

with open('/content/drive/MyDrive/crowd-risk-lstm/data/shanghaitech_meta.json') as f:
    rows = json.load(f)
anom = [r for r in rows if r['anom'] == 1]

for stride in [3, 2]:
    span = 29 * stride
    usable = [r for r in anom if r['start'] >= span]
    sc = Counter(r['vid'].split('_')[0] for r in usable)
    print(f"\nstride {stride} — {len(usable)} videos across {len(sc)} scenes")
    print(" ", dict(sorted(sc.items())))


stride 3 — 23 videos across 8 scenes
  {'01': 8, '03': 3, '04': 2, '05': 3, '06': 1, '07': 3, '08': 2, '12': 1}

stride 2 — 33 videos across 9 scenes
  {'01': 15, '02': 1, '03': 3, '04': 2, '05': 3, '06': 2, '07': 3, '08': 2, '12': 2}


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install ultralytics -q

import zipfile, yaml, shutil, os
from ultralytics import YOLO

with zipfile.ZipFile('/content/drive/MyDrive/crowd-risk-lstm/data/raw/crowdhuman-1/roboflow.zip') as z:
    z.extractall('/content/crowdhuman')

cfg = {'train':'/content/crowdhuman/train/images',
       'val':'/content/crowdhuman/valid/images',
       'test':'/content/crowdhuman/test/images',
       'nc':1, 'names':['person']}
with open('/content/crowdhuman/data.yaml','w') as f:
    yaml.dump(cfg, f)

base = '/content/drive/MyDrive/crowd-risk-lstm'
os.makedirs(f'{base}/models', exist_ok=True)
shutil.copy(f'{base}/outputs/yolo_training/crowdhuman_yolov8s/weights/best.pt',
            f'{base}/models/finetuned_yolov8s_cctv.pt')

print("=== COCO-pretrained YOLOv8s ===")
mb = YOLO('yolov8s.pt').val(data='/content/crowdhuman/data.yaml', split='test', classes=[0])

print("\n=== Fine-tuned (ours) ===")
mf = YOLO(f'{base}/models/finetuned_yolov8s_cctv.pt').val(data='/content/crowdhuman/data.yaml', split='test')

print(f"\n{'Model':<28}{'P':>8}{'R':>8}{'mAP50':>9}{'mAP50-95':>11}")
print(f"{'YOLOv8s (COCO)':<28}{mb.box.p.mean():>8.3f}{mb.box.r.mean():>8.3f}{mb.box.map50:>9.3f}{mb.box.map:>11.3f}")
print(f"{'YOLOv8s + CrowdHuman':<28}{mf.box.p.mean():>8.3f}{mf.box.r.mean():>8.3f}{mf.box.map50:>9.3f}{mf.box.map:>11.3f}")

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 32.2 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
=== COCO-pretrained YOLOv8s ===
Ultralytics 8.4.116 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLOv8s summary (fused): 72 layers, 11,156,544 parameters, 0 gradients, 28.6 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.0±0.0 ms, read: 39.4±22.5 MB/s, size: 389.5 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
val: Scanning /content/crowdhuman/test/labels... 1937 images, 0 back

In [ ]:
import shutil, os
shutil.rmtree('/content/drive', ignore_errors=True)

from google.colab import drive
drive.mount('/content/drive')

base = '/content/drive/MyDrive/crowd-risk-lstm'
for d in ['models', 'src', 'data']:
    p = f'{base}/{d}'
    print(f"{d}/:", os.listdir(p)[:10] if os.path.exists(p) else "MISSING")

Mounted at /content/drive
models/: ['finetuned_yolov8s_cctv.pt']
src/: ['process_umn.py', 'build_forecast_dataset.py', 'diagnose_speed_distribution.py', 'feature_extractor.py', 'diagnose_leadup_signal.py', 'train_forecast_model.py', '__pycache__', 'process_shanghaitech.py']
data/: ['raw', 'umn_scan.csv', 'umn_groundtruth.csv', 'features', 'forecast_dataset.npz', 'shanghaitech_meta.json']


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
base = '/content/drive/MyDrive/crowd-risk-lstm'
work = '/content/drive/MyDrive/shanghai-job'
os.makedirs(work, exist_ok=True)

for src, dst in [
    (f'{base}/models/finetuned_yolov8s_cctv.pt', 'model.pt'),
    (f'{base}/data/raw/shanghaitech/shanghaitech-campus-dataset-test.zip', 'data.zip'),
    (f'{base}/src/feature_extractor.py', 'feature_extractor.py'),
    (f'{base}/src/process_shanghaitech.py', 'process_shanghaitech.py'),
]:
    shutil.copy(src, f'{work}/{dst}')
    print(f"{dst}  ({os.path.getsize(f'{work}/{dst}')/1e6:.1f} MB)")

print("\ncreated:", work)

Mounted at /content/drive
model.pt  (58.9 MB)
data.zip  (9364.2 MB)
feature_extractor.py  (0.0 MB)
process_shanghaitech.py  (0.0 MB)

created: /content/drive/MyDrive/shanghai-job


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import zipfile, os
base = '/content/drive/MyDrive/crowd-risk-lstm'
work = '/content/drive/MyDrive/shanghai-job'

os.makedirs(f'{base}/data/features/shanghaitech', exist_ok=True)
with zipfile.ZipFile(f'{work}/features_2x2.zip') as z:
    z.extractall(f'{base}/data/features/shanghaitech')

import shutil
shutil.copy(f'{work}/shanghaitech_meta_2x2.csv', f'{base}/data/features/shanghaitech_meta.csv')
print("files:", len(os.listdir(f'{base}/data/features/shanghaitech')))

Mounted at /content/drive
files: 199


In [ ]:
import os
p = '/content/drive/MyDrive/crowd-risk-lstm/data/features/shanghaitech'
print(len(os.listdir(p)), "files")

199 files


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/crowd-risk-lstm
!python3 src/train_shanghaitech_forecast_model.py

Mounted at /content/drive
/content/drive/MyDrive/crowd-risk-lstm
Device: cuda
Model: input_size=16 hidden=64 layers=1 dropout=0.2
HEADLINE_SCENES=['01', '03', '05', '07']  THIN_SCENES=['04', '06', '08', '12']  NON_EVALUABLE_SCENES=['02', '09', '10', '11', '13']
Every fold trains on all scenes except its own held-out scene, including all NON_EVALUABLE_SCENES and every other evaluable/thin scene -- training is never restricted to the evaluation scenes. Inner-validation for early stopping is drawn ONLY from NON_EVALUABLE_SCENES in every fold of every group.

1. HEADLINE: leave-one-scene-out over {01,03,05,07}

fold: held-out scene 01  train=78815 (incl. 13926 from ['02', '09', '10', '11', '13'])  test=31604  CLEAN_PRE positives in this scene: {24: 174, 48: 300, 72: 359}
  saved checkpoint + result to outputs/reports/shanghaitech_fold_headline_01.json
  early stop: best_epoch=0 stopped_at=22 inner_train=64889 inner_val=13926
    k=  0: full AUROC=0.5710 AUPRC=0.1016  |  clean_pre+neg AUROC

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/crowd-risk-lstm
!python3 src/train_shanghaitech_forecast_model.py

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/crowd-risk-lstm
Device: cuda
Model: input_size=16 hidden=64 layers=1 dropout=0.2
HEADLINE_SCENES=['01', '03', '05', '07']  THIN_SCENES=['04', '06', '08', '12']  NON_EVALUABLE_SCENES=['02', '09', '10', '11', '13']
Every fold trains on all scenes except its own held-out scene, including all NON_EVALUABLE_SCENES and every other evaluable/thin scene -- training is never restricted to the evaluation scenes. Inner-validation for early stopping holds out ~15% of videos from EVERY scene in the fold's training pool (video-level, stratified by scene), not just the non-evaluable scenes -- narrowing the validation signal to 2-3 atypical scenes produced best_epoch=0 (no improvement ever recorded) on multiple headline folds.

1. HEADLINE: leave-one-scene-out over {01,03,05,07}

fold: held-out scene 01  train=78815 (incl. 13926 from ['02', '09', '10',